# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prabhaditya003/FlyRank.ai-internship-work-week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Content Refresh Urgency vs. Organic Traffic LiftClaim evaluated: "Refreshing decaying URLs within 30 days yields an average 24% organic traffic recovery over the subsequent 90 days."Label origin: The target label is derived from the delta between post-update 90-day organic sessions and pre-update baseline traffic.Methodology question: Was the 90-day post-update evaluation window strictly out-of-time, and were seasonal macro-trends controlled for? If post-update windows for multiple URLs coincided with seasonal traffic spikes (e.g., Q4 e-commerce surges), the observed lift may reflect macro search volume shifts rather than the intervention itself. A robust validation design requires a concurrent temporal control group of un-refreshed decaying URLs to isolate true intervention effects.Finding 2: SERP Rank Drop Prediction ScoreClaim evaluated: "The model achieves an 0.88 ROC-AUC in predicting top-3 SERP position loss over a 60-day horizon."Label origin: Binary indicator ($1$ if average SERP position drops below position 3 within 60 days, $0$ otherwise).Methodology question: Was the cross-validation split grouped by root domain or client account? If URLs from the same domain are present in both training and test sets under a random split, the model can memorize domain-level site authority, backlink profile baselines, or CMS structure. Grouping by domain/client verifies whether the model generalizes to previously unseen sites rather than memorizing domain specificities.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold

# 1. Resolve File Path Dynamically
possible_paths = [
    "content_refresh_anonymized.csv",
    "../content_refresh_anonymized.csv",
    "../../content_refresh_anonymized.csv",
]

file_path = next((p for p in possible_paths if os.path.exists(p)), None)

if file_path is None:
    raise FileNotFoundError(
        "Please upload 'content_refresh_anonymized.csv' using the left sidebar folder icon in Colab."
    )

df = pd.read_csv(file_path)

# 2. Define Target (Content Decay Indicator) & Group Entity
df["target"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["target"].mean()

honest_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
]

X = df[honest_features].fillna(0)
y = df["target"]
groups = df["client_id"]

# 3. Random 5-Fold Split (Naive / Leaky)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rand_auc, rand_prec, rand_rec = [], [], []

for train_idx, test_idx in kf.split(X, y):
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = rf.predict_proba(X.iloc[test_idx])[:, 1]
    preds = (probs >= 0.5).astype(int)

    rand_auc.append(roc_auc_score(y.iloc[test_idx], probs))
    rand_prec.append(precision_score(y.iloc[test_idx], preds))
    rand_rec.append(recall_score(y.iloc[test_idx], preds))

# 4. Grouped 5-Fold Split by client_id (Honest)
gkf = GroupKFold(n_splits=5)
group_auc, group_prec, group_rec = [], [], []

for train_idx, test_idx in gkf.split(X, y, groups=groups):
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = rf.predict_proba(X.iloc[test_idx])[:, 1]
    preds = (probs >= 0.5).astype(int)

    group_auc.append(roc_auc_score(y.iloc[test_idx], probs))
    group_prec.append(precision_score(y.iloc[test_idx], preds))
    group_rec.append(recall_score(y.iloc[test_idx], preds))

# 5. Output Comparison Table
summary_df = pd.DataFrame(
    {
        "Evaluation Strategy": [
            "Base Rate Baseline",
            "Random 5-Fold CV (Leaky)",
            "Grouped 5-Fold CV (Honest)",
        ],
        "ROC-AUC": [0.5000, np.mean(rand_auc), np.mean(group_auc)],
        "Precision": [base_rate, np.mean(rand_prec), np.mean(group_prec)],
        "Recall": [base_rate, np.mean(rand_rec), np.mean(group_rec)],
    }
)

print(f"Base Rate (Declining Content Ratio): {base_rate:.4f}\n")
print(summary_df.to_string(index=False))

Base Rate (Declining Content Ratio): 0.5421

       Evaluation Strategy  ROC-AUC  Precision   Recall
        Base Rate Baseline 0.500000   0.542067 0.542067
  Random 5-Fold CV (Leaky) 0.799610   0.697913 0.866623
Grouped 5-Fold CV (Honest) 0.703376   0.649495 0.880146


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# 1. Feature Importance Audit
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X, y)

result = permutation_importance(clf, X, y, n_repeats=10, random_state=42)

importance_df = pd.DataFrame(
    {"Feature": X.columns, "Importance_Mean": result.importances_mean}
).sort_values(by="Importance_Mean", ascending=False)

print("Top Feature Importances:")
print(importance_df.head(5).to_string(index=False))
print("\n" + "=" * 50 + "\n")

# 2. Leakage Test with 'trend_pct'
X_leaky = df[list(X.columns) + ["trend_pct"]].fillna(0)
gkf = GroupKFold(n_splits=5)
leaky_auc = []

for train_idx, test_idx in gkf.split(X_leaky, y, groups=groups):
    rf_leak = RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42
    )
    rf_leak.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
    probs = rf_leak.predict_proba(X_leaky.iloc[test_idx])[:, 1]
    leaky_auc.append(roc_auc_score(y.iloc[test_idx], probs))

print(f"ROC-AUC with Leaky Feature (trend_pct): {np.mean(leaky_auc):.4f}")
print(f"ROC-AUC without Leaky Feature (Honest): {np.mean(group_auc):.4f}")
print("\n" + "=" * 50 + "\n")

# 3. Error Case Analysis (Using 'target' and 'trend_direction')
df["pred_proba"] = clf.predict_proba(X)[:, 1]
df["pred_label"] = (df["pred_proba"] >= 0.5).astype(int)

fp = df[(df["target"] == 0) & (df["pred_label"] == 1)].head(2)
fn = df[(df["target"] == 1) & (df["pred_label"] == 0)].head(1)
errors = pd.concat([fp, fn])

print("--- 3 Concrete Error Cases ---")
for idx, row in errors.iterrows():
    print(
        f"Content ID: {row['content_id']} | Client: {row['client_id']} | "
        f"Actual: {row['trend_direction']} | Pred Proba: {row['pred_proba']:.3f} | "
        f"Age: {row['content_age_days']} days | Prev Clicks: {row['clicks_prev_30d']}"
    )

Top Feature Importances:
             Feature  Importance_Mean
impressions_prev_30d         0.150807
        avg_position         0.052117
    content_age_days         0.050627
     clicks_prev_30d         0.025223
   sessions_prev_30d         0.025040


ROC-AUC with Leaky Feature (trend_pct): 1.0000
ROC-AUC without Leaky Feature (Honest): 0.7034


--- 3 Concrete Error Cases ---
Content ID: content_a5a2fbc76336 | Client: client_8527a891e2 | Actual: stable | Pred Proba: 0.695 | Age: 238 days | Prev Clicks: 0
Content ID: content_9d548144b06d | Client: client_6208ef0f77 | Actual: stable | Pred Proba: 0.697 | Age: 118 days | Prev Clicks: 0
Content ID: content_a1fb4e703a9e | Client: client_4e07408562 | Actual: down | Pred Proba: 0.487 | Age: 445 days | Prev Clicks: 1


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Claim 1: "Our machine learning model guarantees a 35% traffic boost by automatically identifying and fixing decaying URLs."

Rewritten Safe Claim: "In historical evaluations, pages flagged for refresh exhibited an observed 18% to 24% directional traffic improvement relative to un-refreshed controls, serving as a decision-support indicator for content prioritization."

Original Claim 2: "The algorithm accurately predicts exact SERP rank drops 60 days in advance."

Rewritten Safe Claim: "Under domain-grouped cross-validation, the model demonstrated a measured 0.72 ROC-AUC in ranking URLs by 60-day position decay risk relative to the naive baseline."

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.